# 05 · 报告与对比

**目标**：生成最终的对比报告 —— 5 条净值曲线、衰减率分布、稳健度热图、组合指标对比表。

**输入**：`final_baselines.json`、`wfa_results.csv`、`robustness_score.csv`、`prices_daily.parquet`

**输出**：
- `report_nav_compare.png` — 净值曲线对比
- `report_decay_dist.png` — 衰减率分布
- `report_robustness.png` — 稳健度热图
- `report_summary.csv` — 组合指标对比

In [ ]:
# ============================================================
# cell 0: imports + 全局参数
# ============================================================
from jqdata import *            # 聚宽 magic
import sys, json, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import pandas as pd
import numpy as np

PROJ = Path('/Users/huhao/src/codesnip/python/ai/028-jukuan').resolve()
sys.path.insert(0, str(PROJ))

from etf_portfolio.data_loader import load_parquet
from etf_portfolio.reporting import (
    plot_nav_compare, plot_decay_distribution,
    plot_robustness_heatmap, summarize_baselines,
)

OUTPUT_DIR = PROJ / 'etf_portfolio' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================
# cell 1: 加载 final_baselines + 价格
# ============================================================
with open(OUTPUT_DIR / 'final_baselines.json', encoding='utf-8') as f:
    baselines = json.load(f)
print('已加载组合:', list(baselines.keys()))

prices = load_parquet(OUTPUT_DIR / 'prices_daily.parquet')
ret_daily = prices.pct_change().dropna()

In [ ]:
# ============================================================
# cell 2: 计算每个组合的日收益（统一测试期）
# ============================================================
port_returns = {}
for name, bl in baselines.items():
    weights = {h['code']: h['weight'] for h in bl['holdings']}
    # 只保留候选池内的 ETF（候选 ETF 必须全部有数据）
    valid_codes = [c for c in weights if c in ret_daily.columns]
    if not valid_codes:
        continue
    # 使用每个组合最早能凑齐权重的日期起点（向后兼容）
    sub = ret_daily[valid_codes].dropna()
    port_ret = sum(weights[c] * sub[c] for c in valid_codes)
    port_returns[name] = port_ret
print(f'成功构造 {len(port_returns)} 个组合的日收益序列')

In [ ]:
# ============================================================
# cell 3: 净值曲线对比（核心图）
# ============================================================
plot_nav_compare(
    port_returns,
    title='稳健 ETF 组合净值曲线对比',
    save_path=OUTPUT_DIR / 'report_nav_compare.png',
    figsize=(14, 7),
    log_scale=False,
)

In [ ]:
# ============================================================
# cell 4: 组合指标对比表
# ============================================================
summary = summarize_baselines(port_returns)
summary

In [ ]:
summary.to_csv(OUTPUT_DIR / 'report_summary.csv', index=False, encoding='utf-8-sig')
print('已写入 report_summary.csv')

In [ ]:
# ============================================================
# cell 5: 衰减率分布（WFA 框架输出）
# ============================================================
wfa_metrics = pd.read_csv(OUTPUT_DIR / 'wfa_results.csv', parse_dates=['date'])
plot_decay_distribution(
    wfa_metrics,
    save_path=OUTPUT_DIR / 'report_decay_dist.png',
)

In [ ]:
# ============================================================
# cell 6: 稳健度评分热图
# ============================================================
robust_score = pd.read_csv(OUTPUT_DIR / 'robustness_score.csv', index_col=0)
plot_robustness_heatmap(
    robust_score,
    top_n=15,
    save_path=OUTPUT_DIR / 'report_robustness.png',
)

In [ ]:
# ============================================================
# cell 7 (新): 路线 A 动态调仓组合 vs 路线 B 静态组合 vs 硬编码基线
# ============================================================
# 路线 A：每个调仓日用过去 60 月数据求最优权重，持仓 1 月，再平衡
# 这其实就是 wfa 里 OOS 期间的拼接：从 oos_returns.csv 把 85 个 OOS 片段按时间拼起来
import numpy as np

oos_raw = pd.read_csv(
    OUTPUT_DIR / 'oos_returns.csv',
    header=[0, 1],
    index_col=0,
    parse_dates=True,
)
# oos_raw.columns 是 MultiIndex (objective, rebal_date)
# 每行（日期）只在一个 rebal_date 列下有值 → 拼接 = 对每行取该列的非空值
dynamic_returns = {}
for obj in oos_raw.columns.levels[0]:
    sub = oos_raw[obj]                     # DataFrame: index=日期, columns=rebal_date
    # 拼接：每行取非空值（理论上每行只有一个非空）
    s = sub.apply(lambda r: r.dropna().iloc[0] if r.notna().any() else np.nan, axis=1)
    dynamic_returns["路线A_" + obj] = s.dropna()

# 合并：把 5 个原组合 + 3 个路线 A 动态目标组合拼成一张图
all_returns = {}
all_returns.update(port_returns)             # 5 个原组合（来自 cell 3）
all_returns.update(dynamic_returns)          # 3 个路线 A 动态目标组合

# 画对比图
fig, ax = plt.subplots(figsize=(15, 8))
for name, ret in all_returns.items():
    if ret is None or ret.empty:
        continue
    nav = nav_curve(ret)
    x_vals = nav.index.to_pydatetime()
    ax.plot(x_vals, nav.values, label=name, linewidth=1.5 if '路线A' not in name else 1.2,
            linestyle='-' if '路线A' not in name else '--',
            alpha=0.85)

ax.set_title('路线 A 动态调仓 vs 路线 B 静态 vs 硬编码基线（9.7 年）')
ax.set_xlabel('日期')
ax.set_ylabel('累计净值（起点 1.0）')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'report_route_A_compare.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# cell 7: 各组合的滚动 12 月夏普（稳定性观察）
# ============================================================
import matplotlib.pyplot as plt

def rolling_sharpe(ret, window=252, rf=0.025):
    daily_rf = rf / 252
    excess = ret - daily_rf
    return (excess.rolling(window).mean() / excess.rolling(window).std()) * np.sqrt(252)

fig, ax = plt.subplots(figsize=(14, 5))
for name, ret in port_returns.items():
    rs = rolling_sharpe(ret, window=252)
    ax.plot(rs.index, rs.values, label=name, linewidth=1.2)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('滚动 12 个月夏普比率')
ax.set_xlabel('日期')
ax.set_ylabel('夏普比率')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'report_rolling_sharpe.png', dpi=120, bbox_inches='tight')
plt.show()

## 最终结论

全部报告产物已落盘到 `etf_portfolio/outputs/`：

| 文件 | 内容 |
|------|------|
| `report_nav_compare.png` | 所有组合的累计净值曲线对比 |
| `report_decay_dist.png`   | WFA 衰减率分布（按目标） |
| `report_robustness.png`   | 稳健度评分热图 |
| `report_rolling_sharpe.png` | 滚动 12 月夏普（看稳定性） |
| `report_summary.csv`     | 组合指标对比表 |
| `final_baselines.json`   | 最终组合清单（3 硬编码 + 路线 B 等权 + 路线 B 风险平价） |

查看 `ETF组合优化方案.md` 了解完整方法论、操作指南与结果解读。